# 06 — ApuraMandato: núcleo de gastos, fornecedores e sanções

Execução offline com amostra CEAP e sanções sintéticas. Não comprova HTTP real nem qualidade do LLM online. Fontes eleitorais e emendas permanecem fora do fluxo principal.

In [1]:
from pathlib import Path
import os,sys
RAIZ=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
os.chdir(RAIZ);sys.path.insert(0,str(RAIZ))
os.environ["MODO_OFFLINE"]="true"
os.environ["LANGCHAIN_TRACING_V2"]="false"
os.environ["LANGSMITH_TRACING"]="false"
from src import config
config.MODO_OFFLINE=True
from scripts.prepare_demo import preparar
preparar(RAIZ/"data/demo/apuramandato.db")
print("Banco demo isolado; sanções fictícias.")

/home/laltuf/Infnet/projeto-agentes-langchain/data/demo/apuramandato.db
Banco demo isolado; sanções fictícias.


## SQL somente leitura

In [2]:
from src.data import consultar_ceap
print(consultar_ceap.invoke({"query_sql":"SELECT nome_parlamentar,ROUND(SUM(valor),2) AS total FROM despesas_ceap WHERE ano=2025 GROUP BY nome_parlamentar ORDER BY total DESC LIMIT 3","anos":[2025]}))

[AUDITORIA SQL] SELECT nome_parlamentar,ROUND(SUM(valor),2) AS total FROM despesas_ceap WHERE ano=2025 GROUP BY nome_parlamentar ORDER BY total DESC LIMIT 3
3 registros: [{'nome_parlamentar': 'Tabata Amaral', 'total': 504994.08}, {'nome_parlamentar': 'Nikolas Ferreira', 'total': 389263.25}, {'nome_parlamentar': 'Felipe Carreras', 'total': 33100.63}]


## Conversa: gasto → fornecedores → cadastro → sanções
O cadastro utiliza fixture e as sanções foram importadas de arquivos sintéticos.

In [3]:
from src.agent import montar_agente,trajetoria
agente=montar_agente(com_rag=False,escopo_ampliado=False)
cfg={"configurable":{"thread_id":"nucleo06"}}
for pergunta in ["Quanto Tabata Amaral gastou com combustível em 2025?","Quais os maiores fornecedores dela?","Consulte o cadastro do CNPJ 11.222.333/0001-81.","Essa empresa aparece no CEIS ou CNEP?"]:
    estado=agente.invoke({"messages":[("user",pergunta)]},config=cfg)
    print(pergunta,"\n",estado["messages"][-1].content)
    print("Ferramentas:",[p["ferramenta"] for p in trajetoria(estado) if p["tipo"]=="decisao"])

Quanto Tabata Amaral gastou com combustível em 2025? 
 Tabata Amaral gastou R$ 48.808,83 com combust em 2025 (201 lancamentos). Maiores categorias: Combustíveis E Lubrificantes.: R$ 48.808,83. Fonte: Cota para o Exercício da Atividade Parlamentar (CEAP), 2025.
Ferramentas: ['buscar_deputado', 'listar_despesas']
Quais os maiores fornecedores dela? 
 **Tabata Amaral — 2025 até 2025**

- POSTO BATALHA LTDA — CNPJ 43731926000160: R$ 17.552,73

- 093 - PISTAO SUL - CASCOL COMBUSTIVEIS PARA VEICULOS LTDA — CNPJ 00306597009315: R$ 8.954,70

- SANGIRARDI COMERCIO VAREJISTA DE COMBUSTIVEIS LTD — CNPJ 28241502000123: R$ 3.867,89

- AUTO POSTO PORTAL DA ALDEIA LTDA — CNPJ 06069159000149: R$ 3.482,32

- AUTO POSTO FONTOURA LTDA — CNPJ 05471787000193: R$ 2.527,25

- PLAYTIME COMERCIAL DE COMBUSTIVEIS LTDA — CNPJ 35807993000192: R$ 1.906,11

- POSTO DE COMBUSTIVEIS MLN BRIGADEIRO LTDA — CNPJ 57740782000112: R$ 967,93

- PORTAL DA GRANJA — CNPJ 20495475000103: R$ 806,47

- GROENLANDIA AUTO POSTO LT —

## Recuperação: exceção de hospedagem no Distrito Federal

In [4]:
from src.data import ferramenta_regras_cota
trechos=ferramenta_regras_cota().invoke({"query":"A cota pode pagar hospedagem em Brasília?"})
assert "hospedagem, exceto do parlamentar no Distrito Federal" in trechos
print(trechos[:1600])

Fonte: Ato da Mesa 43/2009 (consolidado)
k) serviços de limpeza e higienização prestado por empresa especializada, incluído o fornecimento dos insumos necessários à execução desses serviços. (Alínea acrescida pelo Ato da Mesa nº 196, de 27/5/2025)

V - assinatura de publicações;

VI - fornecimento de alimentação do parlamentar;

VII - hospedagem, exceto do parlamentar no Distrito Federal;

VIII - outras despesas com locomoção, contemplando:

a) locação ou fretamento de aeronaves;

Fonte: Ato da Mesa 43/2009 (consolidado)
§ 3º Além da cota estabelecida no Anexo deste Ato, o Deputado fará jus ao reembolso de até quatro trechos de bilhetes aéreos, ida e volta, por mês, tendo como destino o Distrito Federal e como origem o respectivo estado de representação ou, desde que autorizado pela Terceira-Secretaria, aeródromo situado em outra unidade da federação e próximo à residência do parlamentar. (Parágrafo acrescido pelo Ato da Mesa nº 270, de 19/1/2023, em vigor em 1º/2/2023)

Fonte: Ato da 

## Limites de escopo e segurança

In [5]:
estado=agente.invoke({"messages":[("user","Quais bens o deputado declarou ao TSE em 2022?")]},config=cfg)
print(estado["messages"][-1].content)
print(consultar_ceap.invoke({"query_sql":"DELETE FROM despesas_ceap"}))

Este fluxo consulta gastos da CEAP, regras da cota, fornecedores e sanções. Dados eleitorais e emendas estão fora do escopo principal. Posso apurar despesas parlamentares por deputado e período.
Erro de seguranca: apenas consultas SELECT sao permitidas.


## Avaliação e defesa

`devbox run validate` testa nove passos sobre o banco configurado e compara valores com referências desse banco. A amostra deste notebook não tem todos os anos.

`devbox run validate-online` usa modelo real e BrasilAPI. `scripts/evaluate_mandato.py --online --langsmith` registra o experimento final. Chamada HTTP real, avaliação online e gravação pessoal precisam ser produzidas com os serviços disponíveis; fixtures não as substituem.